
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lab - Building Multi-Source E-commerce Pipeline with Spark Declarative Pipelines

## Lab Scenario

You are a data engineer working for an e-commerce company that receives orders from two channels: their website and mobile app. These channels store data in different formats and may have some similarities and differences in how they capture order information. Your role is to build a streaming data pipeline using Spark Declarative Pipelines that can process data from these different channels and generate weekly revenue reports for business stakeholders.

This lab will demonstrate advanced techniques including multi-flow ingestion, stream-static joins, data quality expectations, and materialized views for analytics.

## A. Classroom Setup
Follow the cells below to set up your workspace for the lab.

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size:1.1em;">
    Option 1 - Databricks Academy Provided Workspace (Vocareum Workspace)
  </strong>
  <div style="color:#333;">

- If you are running this notebook in a <strong>Databricks Academy provided Vocareum workspace</strong>, your Unity Catalog catalog is already created for you.

- Your catalog name matches your Vocareum username and looks like:
    <strong>labuser12345</strong> (series of unique numbers)
  </div>
</div>


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size:1.1em;">
    Option 2 - Other Workspaces or Databricks Free Edition
  </strong>
  <div style="color:#333;">

- If you are running this notebook in your own Databricks workspace or Databricks Free Edition, the setup will
<strong>create a Unity Catalog catalog and schema for you</strong>.
  - **Create catalog permission is required.**

- The catalog name is derived from your Databricks username and follows this pattern: <strong>labuser_username</strong>
  </div>
</div>

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Do Not Run in Production Environments</strong>
  <div style="color:#333;">
  <ul>
      <li>Only run this notebook in <strong>development or sandbox workspaces</strong>.</li>
      <li>Do not run this in production environments. The setup script creates a catalog and schemas in your workspace.</li>
  </ul>
  </div>
</div>

### A1. Configure Your Catalog and Schema

Run the cell below to initialize your environment. This setup step does the following:
    - **Assumes you have permission to create a catalog** when running outside of a Databricks provided Vocareum workspace
    - Creates three schemas in your specified catalog:
        - **lab_1_bronze**
        - **lab_2_silver**
        - **lab_3_gold**
    - Creates `ops` and `source` volumes in your **YOUR_LABUSER_CATALOG.lab_1_bronze** schema, and adds sample files to your volumes.
    - Verifies your selected compute environment

    This ensures that all schemas, tables and objects are created in your catalog.

> **Important:** You must have permission to create catalogs in your own non-Vocareum workspace. If you do not have the required permissions, this step will fail. Review the note below before continuing.


<div style="
  border-left: 4px solid #ff9800;
  background: #fff3e0;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">

  <strong style="display:block; color:#e65100; margin-bottom:6px; font-size: 1.1em;">
    Troubleshooting Setup - Missing Create Catalog Permissions
  </strong>
<details>
  <div style="color:#333;">

If you do not have permission to create a new catalog but already have one available, you can explicitly specify an existing catalog by using the `catalog_forced` argument in the `build_user_catalog_name` function.

This function is defined in the notebook: `./Includes/Classroom-Setup-multiplex`

  </div>
</details>
</div>

In [0]:
%run ./Includes/Classroom-Setup-lab

### A2. Verify Volume Path Configuration

Run the cell below to view the value of the `my_vol_path` variable.

Confirm that the value references your **your-catalog.lab_1_bronze** path. This will be used to dynamically reference your source volumes throughout this lab.

In [0]:
print(my_vol_path)

## B. Explore Source Data Characteristics

Before building our pipeline, let's understand the structure and characteristics of our source data. This e-commerce company receives orders from two different channels, each with its own data format and schema variations.

### B1. Understanding Data File Structure

Our pipeline will process data from multiple sources, each stored in different volumes with specific formats and purposes:


| File                | Destination Volume | Format | Purpose                                 |
|---------------------|-------------------|--------|-----------------------------------------|
| `web_orders_1.csv`  | web_orders        | CSV    | Web channel orders                      |
| `app_orders_1.json` | app_orders        | JSON   | Mobile app orders                       |
| `product_catalog.csv`| ops              | CSV    | Static product reference data           |

### B2. Listing Data in Destination Volumes

Let's examine the files available in each volume to understand our data sources.

#### 1. Operations Volume

The operations volume contains static reference data like product catalogs.

In [0]:
# List files in ops volume
display(spark.sql(f"LIST '{my_vol_path}/ops'"))

#### 2. Web Orders Volume

The web orders volume contains CSV files with order data from the company's website.

In [0]:
# List files in web_orders volume
display(spark.sql(f"LIST '{my_vol_path}/web_orders'"))

####3. App Orders Volume

The app orders volume contains JSON files with order data from the mobile application.

In [0]:
# List files in app_orders volume
display(spark.sql(f"LIST '{my_vol_path}/app_orders'"))

### B3. Analyzing Data Structure and Content

Now let's examine the actual data structure and content from each source to understand schema differences and commonalities.

#### 1. Analyzing Operations Data

The operations volume contains product catalog information that will be used for enriching order data.

In [0]:
%sql
SELECT * 
FROM read_files(
    my_vol_path || '/ops/'
);

#### 2. Analyzing Web Orders Data

Web orders are stored in CSV format and contain web-specific metadata like browser and session information.

In [0]:
%sql
SELECT * 
FROM read_files(
    my_vol_path || '/web_orders/'
);

#### 3. Analyzing App Orders Data

App orders are stored in JSON format and contain app-specific metadata like app version and device model.

In [0]:
%sql
SELECT * 
FROM read_files(
    my_vol_path || '/app_orders/'
);

#### Checkpoint - Data Exploration

Confirm the following data counts:
- Web orders: Rows loaded from `web_orders_1.csv`
- App orders: Rows loaded from `app_orders_1.json`
- Product catalog: 50 products

**TROUBLESHOOTING:** If any file shows 0 rows, verify `my_vol_path` is correct and all files were uploaded successfully.

## C. Create the Spark Declarative Pipeline

Now we'll create our Spark Declarative Pipeline using the Apache Spark™ Pipelines Editor to process data from multiple sources.

### C1. Enable the Apache Spark™ Pipelines Editor

Complete the following steps to confirm or enable the **Apache Spark™ Pipelines Editor**:

1. In the top-right corner of the workspace, select your **account icon** ![Account Icon](https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/account_icon.png) (*Your icon letter will differ*).  

2. Right-click **Settings** and choose **Open link in new tab**.  

3. In the left sidebar, select **Developer** under **User**.  

4. In the **Experimental features** section, locate **Apache Spark™ Pipelines Editor** and toggle it **on**.

### C2. Create a Apache Spark™ Spark Declarative Pipeline

Complete the following steps to create your Spark Declarative Pipeline using the Apache Spark™ Pipelines Editor:

1. In the main navigation pane, right-click **Jobs & Pipelines** and select **Open link in New Tab**.  

2. In the new tab, select **Create → ETL Pipeline**.  

   **NOTE:** If prompted to **Try the new Apache Spark™ Pipelines Editor**, choose **Enable Apache Spark™ Pipelines Editor**. This appears only if you did not complete the previous step.  

3. Configure the pipeline settings:
   - **Pipeline Name**: `lab_ecommerce_yourname`
   - **Default Catalog**: `YOUR_LABUSER_CATALOG`
   - **Default Schema**: `lab_1_bronze`  
   **NOTE:** Clear the selected schema using the cross icon to view all schemas.

4. **Rename pipeline components**:
   - Rename the **transformations** folder to `ecommerce_pipeline`
   - Rename **my_transformation.py** file to `bronze_ingestion.sql`

5. Leave the **Apache Spark™ Pipelines Editor** page open for the next steps.

### C3. Configure Pipeline Parameters

Pipeline parameters allow us to dynamically reference our source volumes across different environments.

1. Run the cell below to retrieve the key-value pairs needed to set your pipeline configuration parameters for the source volumes.

In [0]:
config_parameters = [
    ("web_orders_source",      f"{my_vol_path}/web_orders"),
    ("app_orders_source",      f"{my_vol_path}/app_orders"),
    ("product_catalog_source", f"{my_vol_path}/ops"),
]

print("=" * 65)
print("  Add these as Configuration Parameters in your Pipeline:")
print("=" * 65)
for key, value in config_parameters:
    print(f"  Key  : {key}")
    print(f"  Value: {value}")
    print()

2. Copy the paths above and add each one as a configuration parameter in your **Spark Declarative Pipeline**.

This will allow your pipeline to reference each volume through parameters.

1. Select **Settings** in your pipeline tab  

2. Under **Configuration**, select **Add configuration**

3. For each **Key**, enter the key name shown above  

4. For each **Value**, enter the corresponding volume path  

5. Select **Save**

**NOTE:** For more details on configuration parameters, see the Databricks documentation: [Use parameters with Apache Spark™ Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/parameters)

## D. Bronze Layer — Multi-Flow Ingestion

We have orders data coming from two sources: **web and app**. The Bronze layer will use multi-flow ingestion to combine data from both sources into a unified table.

To build the Bronze layer, follow these three steps:
1. Create a Bronze table that can accept data from both sources.
2. Create a flow for web orders data.
3. Create a flow for app orders data.

### D1. Understand the Multi-Flow Design

Both channels share core order columns but differ in channel-specific metadata:

| Column | Web CSV | App JSON | Bronze Handling |
|--------|---------|----------|----------------|
| **order_id**, **customer_id**, **sku** | Yes | Yes | Both flows |
 **discount_code** | Yes | Yes | Schema hint applied |
 **browser**, **session_id** | Yes | No | Web-only; NULL for app |
 **app_version**, **device_model** | No | Yes | App-only; NULL for web |

### D2. Creating Bronze Table

Create a **Bronze** streaming table that acts as a single raw landing table for both web and app orders using `CREATE OR REPLACE STREAMING TABLE`.

**Requirements:**
- Create the table in the **lab_1_bronze** schema and name the table **combined_orders_raw**.  
- Include:
  - All common columns.
  - Source‑specific columns.  
  - Future‑facing columns that may not exist yet in current files (`discount_code`).  
- Add metadata columns:
  - **source_file** (file path or logical source name)  
  - **file_mod_time** (`TIMESTAMP`)  
  - **ingestion_time** (`TIMESTAMP`)  
- Use `STRING` for all business columns, and `TIMESTAMP` only for the time metadata columns.   
- In `TBLPROPERTIES`, set `pipelines.reset.allowed = false`.  

**To Do:**
Using these requirements, write the full `CREATE OR REPLACE STREAMING TABLE` statement, including all columns, data types, `COMMENT`, and `TBLPROPERTIES`.

In [0]:
## Create the Bronze streaming table for combined orders
## Use the requirements specified above

%sql
<FILL_IN>

##### CODE ANSWER - Bronze Table Structure

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
----------------------------------------------------------------
-- STEP 1: Define the unified Bronze streaming table
-- All business columns stored as STRING for schema flexibility.
-- ----------------------------------------------------------------
CREATE OR REPLACE STREAMING TABLE lab_1_bronze.combined_orders_raw
(
  order_id          STRING,
  order_date        STRING,
  customer_id       STRING,
  sku               STRING,
  model             STRING,
  category          STRING,
  order_type        STRING,
  channel           STRING,
  quantity          STRING,
  unit_price        STRING,
  discount_code     STRING,
  discount_amount   STRING,
  total_amount      STRING,
  payment_method    STRING,
  order_status      STRING,
  city              STRING,
  region            STRING,
  ship_date         STRING,
  browser           STRING,
  session_id        STRING,
  os                STRING,
  app_version       STRING,
  device_model      STRING,
  source_file       STRING,
  file_mod_time     TIMESTAMP,
  ingestion_time    TIMESTAMP
)
COMMENT "Unified Bronze streaming table - web and app orders combined via multi-flow ingestion."
TBLPROPERTIES (
  'pipelines.reset.allowed' = false
);
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### D3. Creating Flows for Each Source

Now that our Bronze table is ready, we will create two flows: one for app orders and another for web orders. Each flow will ingest data into our streaming Bronze table.

####1. App Orders Flow

Now, we are going to create a flow for ingesting App Orders:
- Create a flow to ingest **JSON** app orders using `read_files`.
- Cast all business fields to `STRING` for schema flexibility.
- Add metadata columns (`source_file`, `file_mod_time`) and `ingestion_time`.
- Insert data into the unified Bronze table using `INSERT INTO ... BY NAME` to align columns by name.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
-- ----------------------------------------------------------------
-- STEP 2: App Orders Flow
-- Reads JSON files from the app_orders volume.
-- Web-specific columns set to NULL.
-- ----------------------------------------------------------------
CREATE FLOW app_orders_flow
AS INSERT INTO lab_1_bronze.combined_orders_raw BY NAME
SELECT
  CAST(order_id        AS STRING)  AS order_id,
  CAST(order_date      AS STRING)  AS order_date,
  CAST(customer_id     AS STRING)  AS customer_id,
  CAST(sku             AS STRING)  AS sku,
  CAST(model           AS STRING)  AS model,
  CAST(category        AS STRING)  AS category,
  CAST(order_type      AS STRING)  AS order_type,
  CAST(channel         AS STRING)  AS channel,
  CAST(quantity        AS STRING)  AS quantity,
  CAST(unit_price      AS STRING)  AS unit_price,
  CAST(discount_code   AS STRING)  AS discount_code,
  CAST(discount_amount AS STRING)  AS discount_amount,
  CAST(total_amount    AS STRING)  AS total_amount,
  CAST(payment_method  AS STRING)  AS payment_method,
  CAST(order_status    AS STRING)  AS order_status,
  CAST(city            AS STRING)  AS city,
  CAST(region          AS STRING)  AS region,
  CAST(ship_date       AS STRING)  AS ship_date,
  CAST(os              AS STRING)  AS os,
  CAST(app_version     AS STRING)  AS app_version,
  CAST(device_model    AS STRING)  AS device_model,
  _metadata.file_name              AS source_file,
  _metadata.file_modification_time AS file_mod_time,
  current_timestamp()              AS ingestion_time
FROM STREAM read_files(
  '${app_orders_source}',
  format => 'json',
  schemaHints => 'discount_code STRING'
);


</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

#### 2. Web Orders Flow

Create a **flow** for web orders that writes into the unified **lab_1_bronze.combined_orders_raw** table using `CREATE FLOW` command.

**Requirements:**
- Read all the columns from the web orders source using `read_files()`
- Select and cast all business columns to `STRING` for schema flexibility.  
- Populate the metadata columns to track lineage:  
  - **source_file** from `_metadata.file_name`  
  - **file_mod_time** from `_metadata.file_modification_time`  
  - **ingestion_time** using `current_timestamp()` 
- Use `schemaHints => 'discount_code STRING'` in `read_files()` so that the **discount_code** column is detected even if it is missing in some CSV files.   
- Use `AS INSERT INTO lab_1_bronze.combined_orders_raw BY NAME` so columns align by name with the Bronze table schema.

**To Do:**

Using these requirements, write the full flow for web orders yourself.

In [0]:
## Create the web orders flow
## Follow the requirements specified above

## <FILL_IN>

##### CODE ANSWER - Web Orders Flow

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
-- ----------------------------------------------------------------
-- STEP 3: Web Orders Flow
-- Reads CSV files from the web_orders volume.
-- App-specific columns set to NULL.
-- ----------------------------------------------------------------
CREATE FLOW web_orders_flow
AS INSERT INTO lab_1_bronze.combined_orders_raw BY NAME
SELECT
  CAST(order_id        AS STRING)  AS order_id,
  CAST(order_date      AS STRING)  AS order_date,
  CAST(customer_id     AS STRING)  AS customer_id,
  CAST(sku             AS STRING)  AS sku,
  CAST(model           AS STRING)  AS model,
  CAST(category        AS STRING)  AS category,
  CAST(order_type      AS STRING)  AS order_type,
  CAST(channel         AS STRING)  AS channel,
  CAST(quantity        AS STRING)  AS quantity,
  CAST(unit_price      AS STRING)  AS unit_price,
  CAST(discount_code   AS STRING)  AS discount_code,
  CAST(discount_amount AS STRING)  AS discount_amount,
  CAST(total_amount    AS STRING)  AS total_amount,
  CAST(payment_method  AS STRING)  AS payment_method,
  CAST(order_status    AS STRING)  AS order_status,
  CAST(city            AS STRING)  AS city,
  CAST(region          AS STRING)  AS region,
  CAST(ship_date       AS STRING)  AS ship_date,
  CAST(browser         AS STRING)  AS browser,
  CAST(session_id      AS STRING)  AS session_id,
  CAST(os              AS STRING)  AS os,
  _metadata.file_name              AS source_file,
  _metadata.file_modification_time AS file_mod_time,
  current_timestamp()              AS ingestion_time
FROM STREAM read_files(
  '${web_orders_source}',
  format      => 'csv',
  header      => true,
  schemaHints => 'discount_code STRING'
);
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### D4. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm that it completes successfully.

2. In the Apache Spark™ Pipelines Editor, explore your pipeline run:
   - Confirm that **200** rows were ingested into the **combined_orders_raw** table from both source volumes.
   - Select the **combined_orders_raw** table and open the **Data** tab to preview all ingested business event records.
   - Observe that web-specific columns like **browser** and **session_id** are populated only for web source records, while app-specific columns like **device_model** and **app_version** are populated only for app source records (and are NULL for web records).

**TROUBLESHOOTING:** If your pipeline does not run successfully, make sure your volumes are created and your configuration parameters are set correctly.

### D5. Confirming Ingestion of Records

Let's verify that our multi-flow ingestion is working correctly by examining the ingested data.

In [0]:
%sql
-- Confirm both flows ingested records
SELECT
  source_file,
  order_type,
  COUNT(*)            AS total_rows,
  MIN(ingestion_time) AS first_ingested,
  MAX(ingestion_time) AS last_ingested
FROM lab_1_bronze.combined_orders_raw
GROUP BY source_file, order_type
ORDER BY source_file;

In [0]:
%sql
-- Verify channel-specific columns are correctly populated or NULL
SELECT
  order_type,
  COUNT(*)             AS total_orders,
  -- Web Specific Columns
  COUNT(browser)       AS browser_count, 
  COUNT(session_id)    AS session_id_count,
  -- App Specific Columns
  COUNT(app_version)   AS app_version_count,
  COUNT(device_model)  AS device_model_count
FROM lab_1_bronze.combined_orders_raw
GROUP BY order_type;

#### Checkpoint - Bronze Layer

| Source            | Expected                                              | Notes                                                                 |
|-------------------|------------------------------------------------------|----------------------------------------------------------------------|
| `web_orders_1.csv`| Web rows ingested|  **browser** and **session_id** populated|
| `app_orders_1.json`| App rows ingested | **app_version** and **device_model** populated|

**TROUBLESHOOTING:** If one flow shows 0 rows, check that the pipeline configuration parameter points to the correct volume path.

## E. Silver Layer — Expectations and Stream-Static Join

The Silver layer will clean and validate our data using expectations, then enrich it with product catalog information through a stream-static join.

### E1. Understand the Data Quality Expectations

Data quality expectations ensure that only valid data flows through our pipeline. Here are the constraints we'll implement:

| Constraint Name | Expectation | Action | Why |
|----------------|-------------|--------|-----|
| `valid_order_id` | `order_id IS NOT NULL` | **FAIL UPDATE** | Primary key — NULL breaks all downstream joins |
| `valid_sku` | `sku IS NOT NULL` | **DROP ROW** | No product reference = unusable for analytics |
| `positive_quantity` | `quantity >= 1` | **DROP ROW** | Quantity of zero or less is not a valid sale |
| `positive_unit_price` | `unit_price > 0` | **WARN** (default) | Flags pricing anomalies without blocking the pipeline |
| `valid_total_amount` | `total_amount >= 0` | **WARN** (default) | Flags negative totals for review |

### E2. Create the Silver SQL File

1. On your **ecommerce_pipeline** folder select the kebab menu and select **Create File**

2. Select the language as **SQL**

3. Name the file `silver_transformation.sql`

### E3. Create the Silver Table

**Instructions:**
- Create a Silver streaming table named **orders_clean** in the **lab_2_silver** schema.
- Use `CREATE OR REFRESH STREAMING TABLE` and select from `STREAM lab_1_bronze.combined_orders_raw`.
- Add data quality constraints using `CONSTRAINT` and `EXPECT` for key columns.
- Use `CLUSTER BY AUTO` for automatic clustering.
- Cast raw fields to correct types with `TRY_CAST`.
- Add a table comment describing the table.

**Requirements:**
- **Table:** **orders_clean**
- **Columns:** Cast fields like **order_date** (`TIMESTAMP`), **quantity** (`INT`), **unit_price**, **discount_amount**, **total_amount** (`DOUBLE`), **ship_date** (`DATE`).
- **Constraints:**
  - `valid_order_id`: **order_id** IS NOT NULL, `ON VIOLATION FAIL UPDATE`
  - `valid_sku`: **sku** IS NOT NULL, `ON VIOLATION DROP ROW`
  - `positive_quantity`: **quantity** >= 1, `ON VIOLATION DROP ROW`
  - `positive_unit_price`: **unit_price** > 0 (WARN)
  - `valid_total_amount`: **total_amount** >= 0 (WARN)

**To Do:**
Using these requirements, write the full code for the Silver table.

##### CODE ANSWER - Silver Table With Expectations and Liquid Clustering Enabled

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
-- ----------------------------------------------------------------
-- STEP 1: Silver clean orders
-- ----------------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE lab_2_silver.orders_clean
(
  -- FAIL: order_id is the primary key
  CONSTRAINT valid_order_id
    EXPECT (order_id IS NOT NULL)
    ON VIOLATION FAIL UPDATE,

  -- DROP: rows without a SKU are unusable for analytics
  CONSTRAINT valid_sku
    EXPECT (sku IS NOT NULL)
    ON VIOLATION DROP ROW,

  -- DROP: quantity less than 1 is not a valid sale
  CONSTRAINT positive_quantity
    EXPECT (quantity >= 1)
    ON VIOLATION DROP ROW,

  -- WARN: flags pricing anomalies; row passes through
  CONSTRAINT positive_unit_price
    EXPECT (unit_price > 0),

  -- WARN: flags negative totals; row passes through
  CONSTRAINT valid_total_amount
    EXPECT (total_amount >= 0)
)
COMMENT "Silver clean orders - type-cast, quality-validated, liquid-clustered."
CLUSTER BY AUTO
AS
SELECT
  order_id,
  TRY_CAST(order_date      AS TIMESTAMP) AS order_date,
  customer_id,
  sku,
  model,
  category,
  order_type,
  channel,
  TRY_CAST(quantity        AS INT)       AS quantity,
  TRY_CAST(unit_price      AS DOUBLE)    AS unit_price,
  discount_code,
  TRY_CAST(discount_amount AS DOUBLE)    AS discount_amount,
  TRY_CAST(total_amount    AS DOUBLE)    AS total_amount,
  payment_method,
  order_status,
  city,
  region,
  TRY_CAST(ship_date       AS DATE)      AS ship_date,
  browser,
  os,
  session_id,
  app_version,
  device_model,
  source_file,
  ingestion_time
FROM STREAM lab_1_bronze.combined_orders_raw;
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### E4. Ingesting Product Catalog

Create a **materialized view** in the Silver layer to hold the static product catalog reference.

**Requirements:**

- Name the view as **lab_2_silver.product_catalog_ref**.   
- Read the `product_catalog.csv` file using `read_files` with:
  - `format => 'csv'`  
  - `header => true`  
- Select and type-cast columns:
  - **sku**  
  - **brand**  
  - **category** as **catalog_category**  
  - **list_price** cast to `DOUBLE` as **list_price**  
  - **is_active** cast to `BOOLEAN` as **is_active**  
- Add a table comment indicating it is a product catalog reference with brand, category, and list price per SKU.

**To Do:**
Using these requirements, write the full statement/code for the view.

##### CODE ANSWER - Creating Product Catalog Reference as Materialized View

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
-- ----------------------------------------------------------------
-- STEP 2: Product catalog as a materialized view (static reference)
-- Used as the static side of the stream-static join below.
-- ----------------------------------------------------------------
CREATE OR REFRESH MATERIALIZED VIEW lab_2_silver.product_catalog_ref
COMMENT "Product catalog reference - brand, category, and list price per SKU."
AS
SELECT
  sku,
  brand,
  category                    AS catalog_category,
  CAST(list_price AS DOUBLE)  AS list_price,
  CAST(is_active  AS BOOLEAN) AS is_active
FROM read_files(
  '${product_catalog_source}',
  format => 'csv',
  header => true
);
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### E5. Join Orders Clean Table with Product Catalog

We will enrich the **orders_clean** table with product catalog attributes using a stream-static join.

**Key Features:**
- Use a stream-static **LEFT JOIN** to retain all orders, even if the **SKU** is missing in the catalog.
- Add product attributes from the catalog:
  - **brand**
  - **catalog_category**
  - **list_price**
  - **is_active** (as **sku_is_active**)
- Calculate **price_vs_catalog** as the difference between the order's **unit_price** and the catalog's **list_price**.

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
-- ----------------------------------------------------------------
-- STEP 3: Enriched orders — stream-static join
-- Streaming side : orders_clean
-- Static side    : product_catalog_ref (re-read on every trigger)
-- LEFT JOIN retains all orders even if the SKU is not in catalog.
-- ----------------------------------------------------------------
CREATE OR REFRESH STREAMING TABLE lab_2_silver.orders_enriched
COMMENT "Enriched Silver orders - stream-static join adds brand, catalog category, and list price."
CLUSTER BY AUTO
AS
SELECT
  o.order_id,
  DATE(o.order_date)                     AS order_date,
  o.customer_id,
  o.sku,
  o.model,
  o.order_type,
  o.channel,
  o.quantity,
  o.unit_price,
  o.discount_code,
  o.discount_amount,
  o.total_amount,
  o.payment_method,
  o.order_status,
  o.city,
  o.region,
  o.ship_date,
  o.source_file,
  p.brand,
  p.catalog_category,
  p.list_price,
  ROUND(o.unit_price - p.list_price, 2)  AS price_vs_catalog,
  p.is_active                             AS sku_is_active
FROM STREAM lab_2_silver.orders_clean AS o
LEFT JOIN lab_2_silver.product_catalog_ref AS p
  ON o.sku = p.sku;
</code></pre>

<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>

### E6. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm it completes successfully.

2. In the Apache Spark™ Pipelines Editor, review your pipeline run:

   - Confirm that **0** new records were ingested into **combined_orders_raw**.

   - **orders_clean** should show **192** records, with 2 expectations met and 3 unmet.

   - Hover over **orders_clean** in the pipeline graph to see **8** records dropped and **8** records with warnings. Click on "Expectations" for details:

     - **8** records dropped due to the `positive_quantity` constraint.
     - **10** records triggered warnings for the `positive_unit_price` constraint.
     -  **2** records failed both constraints, so the graph shows **8** failed for `positive_unit_price`, but actually **10** failed.

   - **product_catalog_ref** should have **50** records.

   - **orders_enriched** should have joined columns from **product_catalog_ref** and also show **192** records.

**TROUBLESHOOTING:** If your pipeline does not run successfully, check that your volumes and configuration parameters are set correctly.

#### Checkpoint - Silver Layer

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/lab/lab_checkpoint_1.png" alt="Silver Layer Checkpoint" width="1200">

### E7. Confirming Transformation of Records

Let's verify that our Silver layer transformations are working correctly.

In [0]:
%sql
SELECT 'orders_clean'    AS table_name, COUNT(*) AS rows FROM lab_2_silver.orders_clean
UNION ALL
SELECT 'orders_enriched' AS table_name, COUNT(*) AS rows FROM lab_2_silver.orders_enriched;

In [0]:
%sql
-- Verify enrichment - brand and list_price populated for known SKUs
SELECT
  order_id, sku, brand, catalog_category,
  unit_price, list_price, price_vs_catalog, sku_is_active
FROM lab_2_silver.orders_enriched
WHERE brand IS NOT NULL
ORDER BY price_vs_catalog DESC
LIMIT 20;

#### Checkpoint — Silver Layer
| Check | Expected |
|-------|----------|
| **orders_clean**  | Bronze rows minus any DROP violations |
| **orders_enriched**  | Equal to orders_clean — LEFT JOIN retains all rows |
| **Expectations tab (pipeline UI)** | 5 constraints shown |
| **Enrichment** | brand, catalog_category, list_price populated for known SKUs |

## F. Gold Layer — Business Analytics Materialized Views

The Gold layer provides business-ready analytics tables optimized for reporting and dashboards.

### F1. Create the Gold Analytics SQL File

1. On your **ecommerce_pipeline** folder select the kebab menu and select **Create File**

2. Select the language as **SQL**

3. Name the file `gold_analytics.sql`

### F2. Daily Revenue by Category Gold View

Create a **Gold-layer materialized view** named **weekly_revenue_by_category** that provides a daily revenue snapshot by category for business stakeholders.

**Requirements:**
- Aggregate sales at the daily level across key business dimensions: date, product category, brand, sales channel, and region.

- Expose core commercial KPIs:
  - **total_orders**: Count of unique orders per group
  - **units_sold**: Total quantity sold
  - **gross_revenue**: Sum of total_amount
  - **total_discounts**: Sum of discount_amount
  - **net_revenue**: Gross revenue minus total discounts
  - **avg_order_value**: Average order value per group
  - **discounted_orders**: Count of orders with a discount code
  - **last_refreshed**: Timestamp of materialized view refresh (`current_timestamp()`)

**To Do:**
Using these requirements, write the full code for the Gold analytics view.

In [0]:
## Create the Gold layer analytics materialized view
## Follow the requirements specified above

%sql
<FILL_IN>

##### CODE ANSWER - Gold Analytics View

<details>
  <summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyBlock()">Copy to clipboard</button>

<pre id="copy-block" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
-- ----------------------------------------------------------------
-- GOLD VIEW: Daily Revenue by Category
-- Audience: Executive dashboards, category managers
-- ----------------------------------------------------------------
CREATE OR REPLACE MATERIALIZED VIEW lab_3_gold.weekly_revenue_by_category
COMMENT "Gold MV: Daily gross revenue, units, and order count by category and channel."
AS
SELECT
    order_date,
    catalog_category                                        AS category,
    brand,
    order_type                                              AS channel,
    region,
    COUNT(DISTINCT order_id)                                AS total_orders,
    SUM(quantity)                                           AS units_sold,
    ROUND(SUM(total_amount), 2)                             AS gross_revenue,
    ROUND(SUM(discount_amount), 2)                         AS total_discounts,
    ROUND(SUM(total_amount) - SUM(discount_amount), 2)     AS net_revenue,
    ROUND(AVG(total_amount), 2)                             AS avg_order_value,
    COUNT(CASE WHEN discount_code IS NOT NULL THEN 1 END)  AS discounted_orders,
    current_timestamp()                                     AS last_refreshed

FROM lab_2_silver.orders_enriched
WHERE order_date IS NOT NULL AND catalog_category  IS NOT NULL
GROUP BY
order_date,catalog_category,brand,order_type,region;
</code></pre>


<script>
function copyBlock() {
  const el = document.getElementById("copy-block");
  if (!el) return;

  const text = el.innerText;

  // Preferred modern API
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackCopy(text);
      });
  } else {
    fallbackCopy(text);
  }
}

function fallbackCopy(text) {
  const textarea = document.createElement("textarea");
  textarea.value = text;
  textarea.style.position = "fixed";
  textarea.style.left = "-9999px";
  document.body.appendChild(textarea);
  textarea.select();
  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    console.error("Fallback copy failed:", err);
    alert("Could not copy to clipboard. Please copy manually.");
  } finally {
    document.body.removeChild(textarea);
  }
}
</script>
</details>

### F3. Run the Full Pipeline and Validate Gold Layer

1. Run the Spark Declarative Pipeline and confirm it completes successfully.
2. In the Apache Spark™ Pipelines Editor, review your pipeline run:

   - Confirm that **0** new records were ingested into any tables of Bronze and Silver layers.
   - Materialized view **weekly_revenue_by_category** should have **192** records

#### Checkpoint - Gold Layer

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/lab/lab_checkpoint_2.png" alt="Gold Layer Checkpoint" width="1200">

In [0]:
%sql
-- Top revenue categories
SELECT
  category,
  SUM(total_orders)            AS total_orders,
  SUM(units_sold)              AS units_sold,
  ROUND(SUM(gross_revenue), 2) AS gross_revenue,
  ROUND(SUM(net_revenue), 2)   AS net_revenue
FROM lab_3_gold.weekly_revenue_by_category
GROUP BY category
ORDER BY gross_revenue DESC;

## G. Incremental Processing

Now we'll simulate incremental data processing by landing additional files and observing how our pipeline handles new data.

### G1. Land Additional Files

Run the function below to automatically land new files into the source locations for both web and app orders.

In [0]:
copy_second_file()

### G2. Listing Available Files

Let's verify that the new files have been added to our source volumes.

In [0]:
# List files in web_orders volume
display(spark.sql(f"LIST '{my_vol_path}/web_orders'"))

In [0]:
# List files in app_orders volume
display(spark.sql(f"LIST '{my_vol_path}/app_orders'"))

### G3. Run and Explore the Pipeline

1. Run the Spark Declarative Pipeline and confirm it completes successfully.

2. In the Apache Spark™ Pipelines Editor, review your pipeline run:

   - Confirm that **200** new records were ingested into **combined_orders_raw**.

   - **orders_clean** should have **191** records, with 3 expectations met and 2 unmet.

   - Hover over **orders_clean** in the pipeline graph to see **9** records dropped and **7** records with warnings. Click on "Expectations" for details:

     - A total of **9** records were dropped: **8** due to the `positive_quantity` constraint and **1** due to the `valid_sku` constraint.
     - **7** records triggered warnings for the `positive_unit_price` constraint.

   - **product_catalog_ref** should have **50** records.

   - **orders_enriched** should have joined columns from **product_catalog_ref** and also show **191** records.

   - **weekly_revenue_by_category** should now show a total of **380** records.

#### Checkpoint - Incremental Processing

<img src="https://files.training.databricks.com/binder/prod_main/advanced-techniques-with-apache-spark-declarative-pipelines-en_us-1.0.3/images/20260828T162008Z/Advanced Techniques with Apache Spark Declarative Pipelines/Includes/images/lab/lab_checkpoint_3.png" alt="Incremental Processing Checkpoint" width="1200">

### G4. Validating the Pipeline

Let's validate that our incremental processing is working correctly by examining the data at each layer.

#### 1. Verify Bronze Cumulative Totals

After both runs, the Bronze layer should contain all records from Run 1 and Run 2 combined.

In [0]:
%sql
SELECT source_file, order_type, COUNT(*) AS rows
FROM lab_1_bronze.combined_orders_raw
GROUP BY source_file, order_type
ORDER BY source_file;

####2. Verify Data Quality Impact

In the first run, **8** records failed. In the second run, **9** records failed. The difference between Bronze and Silver layers is the total dropped records: `8 + 9 = 17`.

In [0]:
%sql
SELECT source_file, order_id, unit_price, sku, quantity FROM lab_1_bronze.combined_orders_raw
MINUS
SELECT source_file, order_id, unit_price, sku, quantity FROM lab_2_silver.orders_clean
ORDER BY source_file;

####3. Data Quality Pass Rate Analysis

Calculate the overall data quality pass rate from Bronze raw to Silver clean.

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM lab_1_bronze.combined_orders_raw)   AS bronze_raw_rows,
  (SELECT COUNT(*) FROM lab_2_silver.orders_clean) AS silver_clean_rows,
  ROUND(
    (SELECT COUNT(*) FROM lab_2_silver.orders_clean) * 100.0 /
    NULLIF((SELECT COUNT(*) FROM lab_1_bronze.combined_orders_raw), 0),
  2) AS pass_rate_pct;

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>